## Fisher scoring

In [2]:
# 1. Импорт библиотек
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# 2. Загрузка очищенных данных
# (Путь '../data/...' корректен, так как блокнот лежит в папке notebooks)
df = pd.read_csv('../data/processed/cleaned_data.csv')

# 3. Разделение на признаки (X) и целевую переменную (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

# 4. Кодирование категориальных признаков (One-Hot Encoding)
X_encoded = pd.get_dummies(X, drop_first=True)

# 5. Разделение на Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Данные готовы. Размер Train: {X_train.shape}, Размер Test: {X_test.shape}")

# --- ИСПРАВЛЕНИЕ ДЛЯ БИБЛИОТЕКИ FISHER-SCORING ---
# Принудительно превращаем все данные в float (числа), чтобы убрать объекты/строки
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# Заменяем бесконечности и NaN на 0 на всякий случай
X_train = X_train.replace([np.inf, -np.inf], 0).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], 0).fillna(0)
# -------------------------------------------------

# 6. Импорт и обучение модели Fisher Scoring
print("Начинаем обучение Fisher Scoring...")
try:
    from fisher_scoring import FisherScoringFocalRegression
    
    # gamma=2 — параметр фокусировки на сложных примерах
    model_fisher = FisherScoringFocalRegression(gamma=2) 
    model_fisher.fit(X_train, y_train)
    
    # 7. Оценка качества модели
    try:
        y_pred_proba_fisher = model_fisher.predict_proba(X_test)[:, 1]
        auc_score = roc_auc_score(y_test, y_pred_proba_fisher)
        print(f"✅ ROC-AUC (Fisher Scoring): {auc_score:.4f}")
    except AttributeError:
        print("⚠️ У модели нет метода predict_proba, считаем метрики по классам...")
        y_pred_fisher = model_fisher.predict(X_test)
        print(classification_report(y_test, y_pred_fisher))
        
except Exception as e:
    print(f"❌ Произошла ошибка при обучении: {e}")
    print("Скорее всего, библиотека fisher-scoring несовместима с вашей версией Python.")

Данные готовы. Размер Train: (5634, 30), Размер Test: (1409, 30)
Начинаем обучение Fisher Scoring...
✅ ROC-AUC (Fisher Scoring): 0.8393


d:\01_Study\Case_02_Telco_Churn\.venv312\Lib\site-packages\fisher_scoring\fisher_scoring_focal.py:230: UserWarning: Near-singular information matrix (condition number: 2.85e+24). Using pseudo-inverse. Results may be unreliable due to multicollinearity or quasi-complete separation.
  beta_new = self.beta + self.invert_matrix(information_matrix) @ score
d:\01_Study\Case_02_Telco_Churn\.venv312\Lib\site-packages\fisher_scoring\fisher_scoring_focal.py:230: UserWarning: Near-singular information matrix (condition number: 6.73e+25). Using pseudo-inverse. Results may be unreliable due to multicollinearity or quasi-complete separation.
  beta_new = self.beta + self.invert_matrix(information_matrix) @ score
d:\01_Study\Case_02_Telco_Churn\.venv312\Lib\site-packages\fisher_scoring\fisher_scoring_focal.py:230: UserWarning: Near-singular information matrix (condition number: 7.64e+25). Using pseudo-inverse. Results may be unreliable due to multicollinearity or quasi-complete separation.
  beta_new